# Claude Haiku 5.5 effort 단계별 정답률, 토큰, 요금

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/haiku_effort.ipynb)

같은 한국어 분류·계산·코딩 과제 40개를 다섯 effort 단계에서 세 번씩 풀어 정답률을 비교합니다.
응답마다 입력·출력 토큰과 걸린 시간을 기록하고, 응답 1,000건당 요금을 달러와 원으로 환산합니다.
유료 Anthropic API 키가 필요합니다. Colab의 열쇠 아이콘에서 `ANTHROPIC_API_KEY`를 등록하고 이 노트북에 접근을 허용하세요.
전체 600회에서 응답당 평균 출력 4,000토큰, 입력 평균을 넉넉히 1,000토큰 이하로 잡으면 출력 1.20달러와 입력 최대 0.06달러로 약 **1.26달러(1,764원)**입니다. 실험 단가와 환율로 계산한 조건부 상한이며 출력이 길어지면 요금이 늘어납니다.

In [ ]:
%pip install -q anthropic

## 실험 설정

`QUICK = True`이면 유형별 첫 두 문제를 low와 max에서 한 번씩 실행합니다(12회). 그래프 언어는 `ko`, `en`, `both` 중에서 선택하세요.

In [ ]:
import ast
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from getpass import getpass
import hashlib
from importlib import import_module
from importlib.util import find_spec
import json
import math
import os
from pathlib import Path
import re
import subprocess
import sys
import tempfile
import time
import unicodedata

import anthropic
import numpy as np
import requests

QUICK = False
CHART_LANG = os.environ.get("CHART_LANG", "ko")
MODEL = "claude-haiku-5-5"
LEVELS = ("low", "medium", "high", "xhigh", "max")
REPEATS = 3
WORKERS = 6
MAX_TOKENS = 32000
MAX_ATTEMPTS = 7  # 첫 호출과 최대 여섯 번의 재시도
# 입력·출력 각각 100만 토큰당 달러
INPUT_RATE, OUTPUT_RATE = 0.10, 0.50
KRW_PER_USD = 1400
KINDS = ("classify", "calc", "code")
LABELS = (
    "결제",
    "환불",
    "배송",
    "계정",
    "기술 오류",
    "제품 문의",
    "기타",
)
CACHE = Path("cache")
RESULTS = Path("results")
FONT_URL = (
    "https://github.com/google/fonts/raw/main/"
    "ofl/nanumgothic/NanumGothic-Regular.ttf"
)
if CHART_LANG not in ("ko", "en", "both"):
    raise ValueError("CHART_LANG은 ko, en, both 중 하나입니다.")
if not math.isfinite(KRW_PER_USD) or KRW_PER_USD <= 0:
    raise ValueError("환율은 양수여야 합니다.")
if REPEATS < 1 or WORKERS < 1:
    raise ValueError(
        "반복 수와 동시 호출 수는 양수여야 합니다."
    )
os.environ.setdefault("MPLCONFIGDIR", str(CACHE / "matplotlib"))
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import (
    LogLocator,
    MaxNLocator,
    NullFormatter,
)

## API 키 준비

Colab 시크릿, 환경 변수, 숨김 입력 순으로 키를 찾습니다. 출력에는 사용한 출처만 표시합니다.

In [ ]:
def configure_key():
    name = "ANTHROPIC_API_KEY"
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
    source = "userdata"
    if not key:
        key = os.environ.get(name)
        source = "env"
    if not key:
        key = getpass("Anthropic API 키 입력: ")
        source = "getpass"
    if not key:
        raise ValueError("빈 API 키입니다.")
    os.environ[name] = key
    print(source)


configure_key()

## 한국어 과제 40개

분류 12개, 계산 16개, 코딩 12개입니다. 모델에는 문제와 답안 형식만 보내며, 정답·테스트·참조 함수는 채점에 사용합니다.

In [ ]:
# 직접 작성한 과제이므로 재배포에 따른 라이선스 문제가 없습니다.
TASKS = {
    "classify": [
        {
            "id": "classify01",
            "prompt": (
                "두 품목을 한 번에 주문했는데 카드 승인 금액이 "
                "주문 합계보다 큽니다. 실제로 얼마가 "
                "청구됐는지 확인해 주세요."
            ),
            "answer": "결제",
        },
        {
            "id": "classify02",
            "prompt": (
                "반품 상자가 도착했다는 알림을 받았습니다. "
                "돌려받을 금액이 아직 입금되지 않았는데 "
                "예정일을 알려 주세요."
            ),
            "answer": "환불",
        },
        {
            "id": "classify03",
            "prompt": (
                "발송 안내를 받은 지 나흘이 지났습니다. "
                "운송장이 계속 집하 전으로 나오는데 물건이 "
                "지금 어디에 있나요?"
            ),
            "answer": "배송",
        },
        {
            "id": "classify04",
            "prompt": (
                "예전에 쓰던 기기를 정리하면서 인증 수단을 "
                "잃었습니다. 기존 계정에 다시 접근하려면 어떤 "
                "절차를 밟아야 하나요?"
            ),
            "answer": "계정",
        },
        {
            "id": "classify05",
            "prompt": (
                "장바구니에서 수량을 바꾸면 화면이 하얗게 "
                "변합니다. 앱을 다시 열어도 같은 현상이 "
                "반복됩니다."
            ),
            "answer": "기술 오류",
        },
        {
            "id": "classify06",
            "prompt": (
                "접이식 선반을 습기가 많은 곳에 두려고 합니다. "
                "받침 부분의 소재와 허용 하중을 알려 주세요."
            ),
            "answer": "제품 문의",
        },
        {
            "id": "classify07",
            "prompt": (
                "지역 창작 모임에서 전시를 준비하고 있습니다. "
                "협업 제안서를 어느 창구에 보내면 되나요?"
            ),
            "answer": "기타",
        },
        {
            "id": "classify08",
            "prompt": (
                "배송비가 붙은 것은 이해했습니다. 주문서대로 "
                "한 번만 냈는데 같은 금액이 두 번 승인된 "
                "이유를 확인해 주세요."
            ),
            "answer": "결제",
        },
        {
            "id": "classify09",
            "prompt": (
                "선물 포장이 찢어져 도착했습니다. 교환할 "
                "생각은 없고 상품을 반납해 지불한 돈을 "
                "돌려받고 싶습니다."
            ),
            "answer": "환불",
        },
        {
            "id": "classify10",
            "prompt": (
                "회원 정보의 기본 주소는 이미 고쳤습니다. 어제 "
                "주문한 상자도 새 주소로 보내 주실 수 있나요?"
            ),
            "answer": "배송",
        },
        {
            "id": "classify11",
            "prompt": (
                "결제 내역을 파일로 내려받으면 합계 열의 "
                "글자가 모두 깨집니다. 화면에서는 정상이라 "
                "내려받기 기능을 고쳐 주세요."
            ),
            "answer": "기술 오류",
        },
        {
            "id": "classify12",
            "prompt": (
                "회원 탈퇴 절차를 진행하고 싶습니다. 어디에서 "
                "탈퇴 신청을 마칠 수 있나요?"
            ),
            "answer": "계정",
        },
    ],
    "calc": [
        {
            "id": "calc_tax",
            "difficulty": "easy",
            "prompt": (
                "가상의 누진 부담금은 처음 80만 원까지 3%, "
                "그다음 120만 원까지 7%, 나머지는 11%이다. "
                "기준액이 253만 원일 때 총 부담금은 몇 원인가?"
            ),
            "answer": 166300,
        },
        {
            "id": "calc_interest",
            "difficulty": "easy",
            "prompt": (
                "원금 123457원에 매년 3.7% 이자를 더한다. 매년 "
                "이자를 더한 직후 원 미만을 버리고 다음 해로 "
                "넘긴다. 4년 직후 잔액은 몇 원인가?"
            ),
            "answer": 142765,
        },
        {
            "id": "calc_elapsed",
            "difficulty": "easy",
            "prompt": (
                "2024년 2월 24일을 0일째로 셀 때 2024년 4월 "
                "3일은 며칠째인가? 그 수에서 정비 기간 5일을 "
                "빼라."
            ),
            "answer": 34,
        },
        {
            "id": "calc_cartons",
            "difficulty": "easy",
            "prompt": (
                "하루 37개씩 9일간 생산한 부품 중 17개는 "
                "검사에서 빠진다. 남은 부품을 한 상자에 최대 "
                "12개씩 담아 모두 보내려면 상자는 최소 몇 "
                "개인가?"
            ),
            "answer": 27,
        },
        {
            "id": "calc_grid",
            "difficulty": "easy",
            "prompt": (
                "(0,0)에서 (7,5)까지 오른쪽이나 위로 한 칸씩 "
                "간다. 반드시 (3,2)를 지나가는 서로 다른 "
                "경로는 몇 개인가?"
            ),
            "answer": 350,
        },
        {
            "id": "calc_draw",
            "difficulty": "easy",
            "prompt": (
                "빨강 5개와 파랑 3개가 든 주머니에서 한 번에 "
                "한 개를 뽑고 되넣는다. 빨강이면 9점, 파랑이면 "
                "1점을 얻을 때 독립적으로 24번 뽑은 총점의 "
                "기댓값은?"
            ),
            "answer": 144,
        },
        {
            "id": "calc_rank",
            "difficulty": "medium",
            "prompt": (
                "A, A, B, C, D를 모두 써 만든 서로 다른 "
                "문자열을 A<B<C<D 순서로 정렬한다. 첫 위치가 "
                "1일 때 CABAD의 위치는?"
            ),
            "answer": 39,
        },
        {
            "id": "calc_order",
            "difficulty": "medium",
            "prompt": (
                "기호 P,Q,R,S,T가 공동 순위 없이 1~5위를 "
                "차지했다. Q는 T 바로 앞이고, P는 S 바로 "
                "뒤이며, R은 Q보다 앞이고 S보다 뒤이다. Q는 몇 "
                "위인가?"
            ),
            "answer": 4,
        },
        {
            "id": "calc_workdays",
            "difficulty": "medium",
            "prompt": (
                "2024년 2월 26일부터 3월 15일까지 양 끝을 "
                "포함한다. 토요일과 일요일을 제외하고, 남아 "
                "있는 날 중 3월 1일과 3월 8일도 제외한다. 남은 "
                "날은 몇 일인가?"
            ),
            "answer": 13,
        },
        {
            "id": "calc_change",
            "difficulty": "medium",
            "prompt": (
                "가치 2, 5, 9인 동전을 각각 제한 없이 써서 "
                "58을 만든다. 동전의 순서는 구별하지 않을 때 "
                "방법은 몇 가지인가?"
            ),
            "answer": 24,
        },
        {
            "id": "calc_reservoir",
            "difficulty": "medium",
            "prompt": (
                "첫날 아침 수조에 물이 317리터 있다. 매일 아침 "
                "보유량의 17%에 해당하는 양을 리터 단위로 "
                "내림해 빼고, 저녁에 23리터를 채운다. 일곱째 "
                "날 저녁의 양은?"
            ),
            "answer": 187,
        },
        {
            "id": "calc_edges",
            "difficulty": "medium",
            "prompt": (
                "정육면체의 각 꼭짓점을 검정 또는 흰색으로 "
                "독립적으로 같은 확률로 칠한다. 양 끝이 다른 "
                "색인 모서리 하나당 7점을 얻고 모두 같은 색인 "
                "면 하나당 8점을 얻는다. 총점의 기댓값은?"
            ),
            "answer": 48,
        },
        {
            "id": "calc_obstacles",
            "difficulty": "hard",
            "prompt": (
                "(0,0)에서 (12,9)까지 오른쪽 또는 위로 한 칸씩 "
                "이동한다. (3,2), (6,5), (8,6)을 모두 피하며 "
                "정확히 (5,4)를 지나는 경로 수는?"
            ),
            "answer": 15312,
        },
        {
            "id": "calc_necklace",
            "difficulty": "hard",
            "prompt": (
                "길이 10인 원형 고리에 붉은 구슬 4개와 흰 구슬 "
                "6개를 끼운다. 이웃한 붉은 구슬이 없고 "
                "마지막과 처음도 이웃한다. 회전만으로 겹치면 "
                "같은 배열이며 뒤집기는 허용하지 않는다. 서로 "
                "다른 배열은 몇 개인가?"
            ),
            "answer": 3,
        },
        {
            "id": "calc_digits",
            "difficulty": "hard",
            "prompt": (
                "0부터 7까지의 숫자 중 서로 다른 여섯 개로 "
                "여섯 자리 정수를 만든다. 맨 앞은 0이 아니며 "
                "수는 7의 배수이고 홀수이다. 이런 정수는 몇 "
                "개인가?"
            ),
            "answer": 1185,
        },
        {
            "id": "calc_schedule",
            "difficulty": "hard",
            "prompt": (
                "시간 구간 [시작,끝)과 보상이 (0,3,6), "
                "(1,4,8), (3,6,7), (4,7,9), (6,8,6), (7,10,8), "
                "(8,11,9), (10,12,8)인 작업이 있다. 겹치지 "
                "않게 정확히 세 개를 고른다. 세 보상의 합에 첫 "
                "작업 종료와 둘째 시작 사이, 둘째 종료와 셋째 "
                "시작 사이의 빈 시간 합의 두 배를 빼면 점수가 "
                "된다. 최대 점수는?"
            ),
            "answer": 25,
        },
    ],
    "code": [
        {
            "id": "code_runs",
            "difficulty": "easy",
            "signature": "def runs(xs):",
            "prompt": (
                "연속해서 같은 정수가 나타난 각 구간의 첫 값만 "
                "남긴 새 리스트를 반환한다."
            ),
            "tests": [
                "assert runs([]) == []",
                "assert runs([0]) == [0]",
                "assert runs([2,2,2]) == [2]",
                "assert runs([1,2,1]) == [1,2,1]",
                "assert runs([-1,-1,0,0,-1]) == [-1,0,-1]",
                "assert runs([3,4,4,5]) == [3,4,5]",
            ],
            "reference": (
                "def runs(xs):\n"
                "    result = []\n"
                "    for x in xs:\n"
                "        if not result or result[-1] != x:\n"
                "            result.append(x)\n"
                "    return result"
            ),
        },
        {
            "id": "code_dial",
            "difficulty": "easy",
            "signature": "def dial(start, moves):",
            "prompt": (
                "0~11까지 표시된 원판에서 start로 출발해 각 "
                "정수만큼 시계 방향으로 움직인다. 음수는 반대 "
                "방향이다. 한 번 움직일 때마다 도착한 눈금을 "
                "순서대로 리스트에 담는다."
            ),
            "tests": [
                "assert dial(0, []) == []",
                "assert dial(11, [1]) == [0]",
                "assert dial(0, [-1, -1]) == [11,10]",
                "assert dial(5, [24,0,-12]) == [5,5,5]",
                "assert dial(3, [4,6,-5]) == [7,1,8]",
                "assert dial(8, [-33]) == [11]",
            ],
            "reference": (
                "def dial(start, moves):\n"
                "    visited = []\n"
                "    for step in moves:\n"
                "        start = (start + step) % 12\n"
                "        visited.append(start)\n"
                "    return visited"
            ),
        },
        {
            "id": "code_checksum",
            "difficulty": "easy",
            "signature": "def checksum(s):",
            "prompt": (
                "ASCII 십진 숫자 문자열에서 오른쪽 끝부터 "
                "차례로 1, 3, 1, 3, …을 곱한 합을 구한다. "
                "여기에 더해 10의 배수로 만드는 최소 음이 아닌 "
                "정수를 반환한다. 빈 문자열의 합은 0이다."
            ),
            "tests": [
                "assert checksum('') == 0",
                "assert checksum('7') == 3",
                "assert checksum('12') == 5",
                "assert checksum('012') == 5",
                "assert checksum('999') == 5",
                "assert checksum('1234') == 2",
            ],
            "reference": (
                "def checksum(s):\n"
                "    total = sum(int(c) * (1 if i % 2 == 0 "
                "else 3) for i, c in enumerate(reversed(s)))\n"
                "    return -total % 10"
            ),
        },
        {
            "id": "code_excess",
            "difficulty": "easy",
            "signature": "def excess(xs, cap):",
            "prompt": (
                "매일의 음이 아닌 정수 사용량 xs에서 일일 한도 "
                "cap을 넘은 부분만 합산한다. 남은 한도는 다음 "
                "날로 이월되지 않는다."
            ),
            "tests": [
                "assert excess([], 5) == 0",
                "assert excess([4], 5) == 0",
                "assert excess([5,5], 5) == 0",
                "assert excess([9,2,8], 5) == 7",
                "assert excess([0,2,3], 0) == 5",
                "assert excess([1,10], 4) == 6",
            ],
            "reference": (
                "def excess(xs, cap):\n"
                "    return sum(max(x - cap, 0) for x in xs)"
            ),
        },
        {
            "id": "code_union_length",
            "difficulty": "medium",
            "signature": "def union_length(spans):",
            "prompt": (
                "정수 반열린 구간 (시작,끝)들의 합집합 길이를 "
                "구한다. 입력은 정렬되지 않을 수 있고 시작과 "
                "끝이 같을 수 있다. 시작은 끝 이하이다."
            ),
            "tests": [
                "assert union_length([]) == 0",
                "assert union_length([(2,2)]) == 0",
                "assert union_length([(1,5),(2,3)]) == 4",
                "assert union_length([(3,7),(0,4)]) == 7",
                (
                    "assert union_length([(-3,0),(0,2),(5,7)]) "
                    "== 7"
                ),
                "assert union_length([(1,6),(1,6),(4,4)]) == 5",
            ],
            "reference": (
                "def union_length(spans):\n"
                "    total, end = 0, None\n"
                "    for a, b in sorted(spans):\n"
                "        total += b - a if end is None else "
                "max(0, b - max(a, end))\n"
                "        end = b if end is None else max(end, "
                "b)\n"
                "    return total"
            ),
        },
        {
            "id": "code_first_window",
            "difficulty": "medium",
            "signature": "def first_window(s, k):",
            "prompt": (
                "문자열에서 서로 다른 문자가 k개인 길이 k짜리 "
                "연속 부분 문자열의 첫 시작 인덱스(0부터)를 "
                "반환한다. 없으면 -1이고 k는 양수이다."
            ),
            "tests": [
                "assert first_window('', 1) == -1",
                "assert first_window('z', 1) == 0",
                "assert first_window('aaaa', 2) == -1",
                "assert first_window('aabc', 3) == 1",
                "assert first_window('abca', 4) == -1",
                "assert first_window('가가나다라', 4) == 1",
            ],
            "reference": (
                "def first_window(s, k):\n"
                "    for i in range(len(s) - k + 1):\n"
                "        if len(set(s[i:i+k])) == k:\n"
                "            return i\n"
                "    return -1"
            ),
        },
        {
            "id": "code_peak_rooms",
            "difficulty": "medium",
            "signature": "def peak_rooms(spans):",
            "prompt": (
                "양의 길이의 반열린 예약 구간 (시작,끝) 목록을 "
                "수용하는 데 필요한 최소 방 수를 반환한다. 한 "
                "예약이 끝나는 순간 같은 방에 다른 예약을 "
                "시작할 수 있다."
            ),
            "tests": [
                "assert peak_rooms([]) == 0",
                "assert peak_rooms([(1,2)]) == 1",
                "assert peak_rooms([(0,2),(2,4)]) == 1",
                "assert peak_rooms([(1,4),(1,4),(1,4)]) == 3",
                "assert peak_rooms([(3,8),(0,5),(4,7)]) == 3",
                "assert peak_rooms([(-4,0),(-2,3),(0,5)]) == 2",
            ],
            "reference": (
                "def peak_rooms(spans):\n"
                "    events = [(a, 1) for a, b in spans] + "
                "[(b, -1) for a, b in spans]\n"
                "    peak = active = 0\n"
                "    for _, delta in sorted(events):\n"
                "        active += delta\n"
                "        peak = max(peak, active)\n"
                "    return peak"
            ),
        },
        {
            "id": "code_ledger",
            "difficulty": "medium",
            "signature": "def ledger(entries):",
            "prompt": (
                "입력 순서대로 (항목명, 정수 증감량)을 "
                "누적한다. 최종 합이 양수인 항목명만 합 "
                "내림차순으로 반환하며, 동점이면 처음 등장한 "
                "순서를 따른다."
            ),
            "tests": [
                "assert ledger([]) == []",
                "assert ledger([('q',-2)]) == []",
                "assert ledger([('x',3),('x',-3)]) == []",
                "assert ledger([('z',1),('a',1)]) == ['z','a']",
                (
                    "assert ledger([('a',-1),('b',3),('a',4)]) "
                    "== ['a','b']"
                ),
                (
                    "assert "
                    "ledger([('a',2),('b',5),('c',1),('c',-4)])"
                    " == ['b','a']"
                ),
            ],
            "reference": (
                "def ledger(entries):\n"
                "    totals = {}\n"
                "    for key, delta in entries:\n"
                "        totals[key] = totals.get(key, 0) + "
                "delta\n"
                "    return sorted((k for k in totals if "
                "totals[k] > 0), key=totals.get, reverse=True)"
            ),
        },
        {
            "id": "code_unpack",
            "difficulty": "medium",
            "signature": "def unpack(s):",
            "prompt": (
                "입력은 영문 소문자 한 글자와 양의 십진 "
                "정수(선행 0 없음)가 번갈아 나오는 문자열이다. "
                "글자를 뒤의 수만큼 반복해 하나로 이어 "
                "반환한다. 빈 입력도 허용한다."
            ),
            "tests": [
                "assert unpack('') == ''",
                "assert unpack('a1') == 'a'",
                "assert unpack('z12') == 'z'*12",
                "assert unpack('a2b3') == 'aabbb'",
                "assert unpack('a1a2b1') == 'aaab'",
                (
                    "assert unpack('b10c2a11') == "
                    "'b'*10+'cc'+'a'*11"
                ),
            ],
            "reference": (
                "def unpack(s):\n"
                "    import re\n"
                "    return ''.join(c * int(n) for c, n in "
                "re.findall(r'([a-z])([0-9]+)', s))"
            ),
        },
        {
            "id": "code_count_sums",
            "difficulty": "hard",
            "signature": "def count_sums(xs, target):",
            "prompt": (
                "정수 리스트의 비어 있지 않은 연속 부분 구간 "
                "중 합이 target인 구간 수를 반환한다. 값이 "
                "같아도 인덱스가 다르면 별도로 센다. 음수와 "
                "0을 허용하며 시간 복잡도는 O(n)이어야 한다."
            ),
            "tests": [
                "assert count_sums([], 0) == 0",
                "assert count_sums([0,0,0], 0) == 6",
                "assert count_sums([1,-1,1,-1], 0) == 4",
                "assert count_sums([-2,1,-2], -1) == 2",
                "assert count_sums([3,3,3], 6) == 2",
                "assert count_sums([0]*10000, 0) == 50005000",
            ],
            "reference": (
                "def count_sums(xs, target):\n"
                "    seen = {0: 1}\n"
                "    count = prefix = 0\n"
                "    for x in xs:\n"
                "        prefix += x\n"
                "        count += seen.get(prefix - target, "
                "0)\n"
                "        seen[prefix] = seen.get(prefix, 0) + "
                "1\n"
                "    return count"
            ),
        },
        {
            "id": "code_chain",
            "difficulty": "hard",
            "signature": "def chain(words):",
            "prompt": (
                "입력 순서를 유지해 단어를 선택한다. 앞 단어의 "
                "마지막 글자가 다음 단어의 첫 글자와 같아야 "
                "한다. 가장 긴 선택의 길이를 반환한다. 단어는 "
                "비어 있지 않고 같은 단어의 서로 다른 출현도 "
                "선택할 수 있다."
            ),
            "tests": [
                "assert chain([]) == 0",
                "assert chain(['xx']) == 1",
                "assert chain(['ab','cd','ef']) == 1",
                "assert chain(['ab','ca','bc']) == 2",
                "assert chain(['ab','bx','bc','cd','dx']) == 4",
                (
                    "assert "
                    "chain(['aa','aa','ab','bb','ba','aa']) == "
                    "6"
                ),
            ],
            "reference": (
                "def chain(words):\n"
                "    best = {}\n"
                "    for word in words:\n"
                "        length = 1 + best.get(word[0], 0)\n"
                "        best[word[-1]] = max(length, "
                "best.get(word[-1], 0))\n"
                "    return max(best.values(), default=0)"
            ),
        },
        {
            "id": "code_min_partition",
            "difficulty": "hard",
            "signature": "def min_partition(xs, k):",
            "prompt": (
                "길이 1 이상의 음이 아닌 정수 리스트를 정확히 "
                "k개의 비어 있지 않은 연속 구간으로 나눈다(1 ≤ "
                "k ≤ 길이). 각 구간 합 중 최댓값을 가능한 한 "
                "작게 만들었을 때 그 값을 반환한다."
            ),
            "tests": [
                "assert min_partition([0], 1) == 0",
                "assert min_partition([2,1,4], 1) == 7",
                "assert min_partition([2,1,4], 3) == 4",
                "assert min_partition([7,2,5,10,8], 2) == 18",
                "assert min_partition([0,0,0,0], 3) == 0",
                "assert min_partition([1,1,8,1,1], 3) == 8",
                "assert min_partition([5,4,3,2,1], 2) == 9",
            ],
            "reference": (
                "def min_partition(xs, k):\n"
                "    low, high = max(xs), sum(xs)\n"
                "    while low < high:\n"
                "        mid = (low + high) // 2\n"
                "        parts, subtotal = 1, 0\n"
                "        for x in xs:\n"
                "            if subtotal + x > mid:\n"
                "                parts += 1\n"
                "                subtotal = 0\n"
                "            subtotal += x\n"
                "        if parts <= k:\n"
                "            high = mid\n"
                "        else:\n"
                "            low = mid + 1\n"
                "    return low"
            ),
        },
    ],
}

## 계산 정답 검증

조합 열거, 날짜 계산, 경로 수 세기 등으로 계산 답안을 재생성해 저장된 정수와 대조합니다.

In [ ]:
from datetime import date, timedelta
from fractions import Fraction
from itertools import combinations, permutations
from math import comb


def answers():
    balance = 123457
    for _ in range(4):
        balance = balance * 1037 // 1000

    water = 317
    for _ in range(7):
        water += 23 - water * 17 // 100

    orders = []
    for positions in permutations("PQRST"):
        p, q, r, s, t = map(positions.index, "PQRST")
        if q + 1 == t and s + 1 == p and s < r < q:
            orders.append(q + 1)
    assert len(orders) == 1

    start, end = date(2024, 2, 26), date(2024, 3, 15)
    holidays = {date(2024, 3, 1), date(2024, 3, 8)}
    dates = (
        start + timedelta(days=i)
        for i in range((end - start).days + 1)
    )

    # 금지점을 피하는 경로의 수를 누적한다.
    def paths(start, end):
        counts = {start: 1}
        for x in range(start[0], end[0] + 1):
            for y in range(start[1], end[1] + 1):
                if (x, y) != start and (x, y) not in {
                    (3, 2),
                    (6, 5),
                    (8, 6),
                }:
                    counts[x, y] = counts.get(
                        (x - 1, y), 0
                    ) + counts.get((x, y - 1), 0)
        return counts[end]

    necklaces = set()
    for reds in combinations(range(10), 4):
        if any((i + 1) % 10 in reds for i in reds):
            continue
        ring = tuple(i in reds for i in range(10))
        necklaces.add(
            min(ring[k:] + ring[:k] for k in range(10))
        )

    jobs = [
        (0, 3, 6),
        (1, 4, 8),
        (3, 6, 7),
        (4, 7, 9),
        (6, 8, 6),
        (7, 10, 8),
        (8, 11, 9),
        (10, 12, 8),
    ]
    scores = []
    for a, b, c in combinations(jobs, 3):
        if a[1] <= b[0] and b[1] <= c[0]:
            idle = b[0] - a[1] + c[0] - b[1]
            scores.append(a[2] + b[2] + c[2] - 2 * idle)

    cube_score = 12 * 7 * Fraction(1, 2) + 6 * 8 * Fraction(
        2, 16
    )
    word_list = sorted(set(permutations("AABCD")))
    numbers = (
        int("".join(p))
        for p in permutations("01234567", 6)
        if p[0] != "0" and p[-1] in "1357"
    )
    return {
        "calc_tax": (800000 * 3 + 1200000 * 7 + 530000 * 11)
        // 100,
        "calc_interest": balance,
        "calc_elapsed": (
            date(2024, 4, 3) - date(2024, 2, 24)
        ).days
        - 5,
        "calc_cartons": (37 * 9 - 17 + 11) // 12,
        "calc_grid": comb(5, 2) * comb(7, 3),
        "calc_draw": 24 * (5 * 9 + 3) // 8,
        "calc_rank": word_list.index(tuple("CABAD")) + 1,
        "calc_order": orders[0],
        "calc_workdays": sum(
            d.weekday() < 5 and d not in holidays for d in dates
        ),
        "calc_change": sum(
            (58 - 9 * n - 5 * m) % 2 == 0
            for n in range(58 // 9 + 1)
            for m in range((58 - 9 * n) // 5 + 1)
        ),
        "calc_reservoir": water,
        "calc_edges": int(cube_score),
        "calc_obstacles": paths((0, 0), (5, 4))
        * paths((5, 4), (12, 9)),
        "calc_necklace": len(necklaces),
        "calc_digits": sum(n % 7 == 0 for n in numbers),
        "calc_schedule": max(scores),
    }


computed = answers()
calc_tasks = TASKS["calc"]
assert len(calc_tasks) == len(computed)
assert {t["id"] for t in calc_tasks} == computed.keys()
for task in calc_tasks:
    assert type(task["answer"]) is int, task["id"]
    assert task["answer"] == computed[task["id"]], task["id"]
print(f"계산 정답 {len(calc_tasks)}건 일치")

## 코딩 참조 해답 자기 테스트

함수 하나를 담은 Python 코드 블록인지 확인한 뒤, 임시 디렉터리의 별도 프로세스에서 각 문제의 테스트를 실행합니다. 제한 시간은 문제당 10초입니다. 모델이 작성한 코드도 같은 방식으로 실행하므로 개인 파일과 드라이브를 연결하지 않은 새 런타임을 사용하세요.

In [ ]:
def code_grade(text, task):
    match = re.fullmatch(
        r"```python[ \t]*\r?\n(.*?)\r?\n```",
        text.strip(),
        re.DOTALL,
    )
    if not match:
        return False, "코드 블록 형식"
    source = match[1]
    try:
        body = ast.parse(source).body
    except (ValueError, SyntaxError, RecursionError):
        return False, "파이썬 구문"
    expected_name = task["signature"].split("(")[0].split()[1]
    if (
        len(body) != 1
        or not isinstance(body[0], ast.FunctionDef)
        or body[0].name != expected_name
    ):
        return False, "함수 정의"
    with tempfile.TemporaryDirectory() as folder:
        directory = Path(folder)
        script = directory / "grade.py"
        # 모든 assert 통과 후에만 완료 파일을 만든다.
        script.write_text(
            source
            + "\n\n"
            + "\n".join(task["tests"])
            + "\nopen('passed', 'x').close()\n",
            encoding="utf-8",
        )
        try:
            result = subprocess.run(
                [sys.executable, str(script)],
                cwd=directory,
                timeout=10,
                check=False,
                env={
                    "HOME": folder,
                    "TMPDIR": folder,
                    "PYTHONNOUSERSITE": "1",
                },
                stdin=subprocess.DEVNULL,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
        except subprocess.TimeoutExpired:
            return False, "10초 초과"
        passed = (
            result.returncode == 0
            and (directory / "passed").exists()
        )
        return passed, "통과" if passed else "테스트 실패"


for task in TASKS["code"]:
    block = f"```python\n{task['reference']}\n```"
    passed, reason = code_grade(block, task)
    print(f"{task['id']}  {len(task['tests'])}개  {reason}")
    assert passed, task["id"]
print(f"코딩 참조 해답 {len(TASKS['code'])}건 통과")

## 채점 기준

분류와 계산은 마지막 줄의 `정답: <값>`을 읽습니다. 계산은 부호·천 단위 쉼표·앞자리 0을 처리하고, 코딩은 참조 해답과 같은 테스트를 모두 통과해야 정답입니다.
오답은 값 틀림, 형식 위반, 답 없음으로 나눕니다. 분류·계산 답의 형식이 어긋났으면 값을 추출해 정답과 일치하는지도 기록합니다.

In [ ]:
WRONG_TYPES = {
    "wrong_value": "값 틀림",
    "format_violation": "형식 위반",
    "no_answer": "답 없음",
}
ANSWER_LINE = re.compile(
    r"(?P<outer>`{1,3}|\*{1,3}|_{1,3}|)[ \t]*"
    r"(?P<label>`{1,3}|\*{1,3}|_{1,3}|)정답"
    r"(?:[ \t]*:(?P=label)|(?P=label)[ \t]*:)"
    r"(?P<value>.*?)(?P=outer)"
)
ANSWER_VALUE_WRAPPER = re.compile(
    r"(?P<marker>`{1,3}|\*{1,3}|_{1,3})"
    r"(?P<value>.+)(?P=marker)"
)


def grade(text, task):
    if task["kind"] == "code":
        return code_grade(text, task)
    lines = text.rstrip().splitlines()
    tail = lines[-1].strip() if lines else ""
    match = re.fullmatch(r"정답:[ \t]*(.+)", tail)
    if not match:
        return False, "마지막 줄 형식"
    value = match[1].strip()
    if task["kind"] == "calc":
        if not re.fullmatch(
            r"[+-]?(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+)", value
        ):
            return False, "정수 형식"
        # 긴 숫자도 int() 변환 제한 없이 비교한다.
        digits = (
            value.replace(",", "").lstrip("+-").lstrip("0")
            or "0"
        )
        value = (
            "-"
            if value.startswith("-") and digits != "0"
            else ""
        ) + digits
    correct = value == str(task["answer"])
    return correct, "통과" if correct else "값 불일치"


def wrong_answer_details(
    text, task, stop_reason, grade_reason
):
    lines = text.rstrip().splitlines()
    last_line = lines[-1] if lines else ""
    details = dict(last_line=last_line, wrong_type="no_answer")
    if task["kind"] == "code":
        if grade_reason in ("테스트 실패", "10초 초과"):
            details["wrong_type"] = "wrong_value"
        elif grade_reason != "코드 블록 형식" or (
            stop_reason != "max_tokens" and re.search(
                r"^[ \t]*(?:```|(?:async[ \t]+)?def[ \t]+)",
                text,
                re.MULTILINE,
            )
        ):
            details["wrong_type"] = "format_violation"
        return details

    match = ANSWER_LINE.fullmatch(last_line.strip())
    if not match:
        return details
    value = match["value"].strip()
    wrapped = ANSWER_VALUE_WRAPPER.fullmatch(value)
    if wrapped:
        value = wrapped["value"].strip()
    if not value:
        return details
    if grade_reason == "값 불일치" and not (
        match["outer"] or match["label"] or wrapped
    ):
        details["wrong_type"] = "wrong_value"
    else:
        value_correct, _ = grade(f"정답: {value}", task)
        details.update(
            wrong_type="format_violation",
            format_value=value,
            format_value_correct=value_correct,
        )
    return details

## API 호출과 응답 캐시

같은 과제·반복 안에서 effort 단계를 교차해 요청합니다. 성공 응답을 `cache/responses/`에 저장하며, 재실행하면 일치하는 캐시의 응답과 당시 소요 시간을 사용해 다시 채점합니다. 실패한 요청은 다음 실행 때 다시 호출합니다.
스트림 읽기 실패, 연결 오류, 호출 한도, HTTP 5xx는 대기 시간을 늘려 가며 최대 여섯 번 재시도합니다.

In [ ]:
# Colab과 로컬 SDK가 사용하는 전송 패키지가 다를 수 있다.
HTTP_TRANSPORTS = tuple(
    import_module(name)
    for name in ("httpx", "httpx2")
    if find_spec(name) is not None
)
STREAM_ERRORS = tuple(
    exc
    for transport in HTTP_TRANSPORTS
    for exc in (
        transport.RemoteProtocolError,
        transport.ReadError,
        transport.ReadTimeout,
    )
)
RETRYABLE_ERRORS = (
    anthropic.RateLimitError,
    anthropic.APIConnectionError,
) + STREAM_ERRORS


def make_prompt(task):
    if task["kind"] == "code":
        return (
            f"{task['signature']}\n{task['prompt']}\n"
            "Python 3.10 표준 라이브러리를 사용할 수 있다. "
            "import는 함수 안에 둔다. "
            "설명 없이 ```python 코드 블록 하나로 "
            "함수만 작성하라."
        )
    instruction = (
        "고객이 요청한 조치에 맞는 범주 하나를 고르라. 범주: "
        + ", ".join(LABELS)
        if task["kind"] == "classify"
        else "정수 하나를 구하라."
    )
    return (
        f"{instruction}\n{task['prompt']}\n"
        "마지막 줄에 `정답: <값>`만 쓰라."
    )


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        dir=path.parent,
        delete=False,
        mode="w",
        encoding="utf-8",
    ) as out:
        temporary = Path(out.name)
        try:
            json.dump(
                data,
                out,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )
            out.write("\n")
            out.close()
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)


def fetch_response(client, prompt, level):
    params = dict(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        output_config={"effort": level},
        messages=[{"role": "user", "content": prompt}],
    )
    for attempt in range(1, MAX_ATTEMPTS + 1):
        start = time.perf_counter()
        try:
            try:
                message = client.messages.create(**params)
            except ValueError as exc:
                if "Streaming is required" not in str(exc):
                    raise
                start = time.perf_counter()
                with client.messages.stream(**params) as stream:
                    message = stream.get_final_message()
            latency = time.perf_counter() - start
            usage = message.usage
            details = getattr(
                usage, "output_tokens_details", None
            )
            return dict(
                text="".join(
                    b.text
                    for b in message.content
                    if b.type == "text"
                ),
                input_tokens=usage.input_tokens,
                output_tokens=usage.output_tokens,
                thinking_tokens=getattr(
                    details, "thinking_tokens", None
                ),
                stop_reason=message.stop_reason,
                latency_s=latency,
                attempts=attempt,
            )
        except (anthropic.APIError, *STREAM_ERRORS) as exc:
            status = getattr(exc, "status_code", 0) or 0
            if attempt == MAX_ATTEMPTS or not (
                isinstance(exc, RETRYABLE_ERRORS)
                or 500 <= status < 600
            ):
                reason = (
                    f"HTTP {status}"
                    if status
                    else type(exc).__name__
                )
                return dict(error=reason, attempts=attempt)
            time.sleep(2 ** (attempt - 1))


def run_one(client, task, level, rep):
    task_id = task["id"]
    record = dict(
        task_id=task_id,
        kind=task["kind"],
        level=level,
        rep=rep,
        correct=False,
        cached=False,
        attempts=0,
        input_tokens=None,
        output_tokens=None,
        thinking_tokens=None,
        latency_s=None,
        stop_reason=None,
        error=None,
    )
    try:
        prompt = make_prompt(task)
        cache = CACHE / "responses"
        path = cache / level / f"{task_id}_{rep}.json"
        identity = [MODEL, MAX_TOKENS, level, rep, prompt]
        signature = hashlib.sha256(
            json.dumps(identity).encode()
        ).hexdigest()
        response = None
        if path.exists():
            saved = json.loads(path.read_text(encoding="utf-8"))
            if saved["signature"] == signature:
                response = saved["response"]
                record["cached"] = True
        if response is None:
            response = fetch_response(client, prompt, level)
            if "error" not in response:
                write_json(
                    path,
                    dict(
                        signature=signature, response=response
                    ),
                )
        record.update(
            {k: v for k, v in response.items() if k != "text"}
        )
        if record["error"] is None:
            text = response["text"]
            record["correct"], record["grade_reason"] = grade(
                text, task
            )
            if not record["correct"]:
                record.update(
                    wrong_answer_details(
                        text, task, record["stop_reason"],
                        record["grade_reason"],
                    )
                )
    except Exception as exc:
        # 한 호출의 오류가 결과 수집을 중단시키지 않는다.
        record["error"] = type(exc).__name__
    return record

## 실험 실행

이 셀에서 유료 호출을 시작합니다. 실행 환경과 과제 수를 먼저 표시합니다.

In [ ]:
levels = ("low", "max") if QUICK else LEVELS
reps = 1 if QUICK else REPEATS
tasks = []
for kind in KINDS:
    batch = TASKS[kind][:2] if QUICK else TASKS[kind]
    tasks.extend(dict(t, kind=kind) for t in batch)
started = datetime.now(timezone.utc).isoformat(
    timespec="seconds"
)
print(f"실행 시각(UTC)  {started}")
print(f"SDK  anthropic {anthropic.__version__}")
print(f"모델  {MODEL}")
print(f"반복  {reps}회    동시 호출  {WORKERS}개")
print("모드  API 실측")
print(
    f"과제  {len(tasks)}건    단계  {len(levels)}개", flush=True
)
jobs = [
    (t, level, rep)
    for rep in range(1, reps + 1)
    for t in tasks
    for level in levels
]
with anthropic.Anthropic(max_retries=0) as client:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [
            pool.submit(run_one, client, *job) for job in jobs
        ]
        rows = [f.result() for f in futures]

## 단계별 집계

정답률의 분모는 실패를 포함한 전체 요청 수입니다. 시간·토큰·기본 환산 요금은 응답을 받은 요청으로 계산하고, 추론 토큰 평균에는 해당 수치가 보고된 응답만 포함합니다.
한 번이라도 `max_tokens`로 끝난 과제는 모든 단계와 반복에서 제외해 별도로 집계합니다. 이 집계의 요금은 남은 전체 요청을 기준으로 환산합니다. 호출 기록과 두 집계는 `results/results.json`에 저장합니다.

In [ ]:
def mean(rows, key):
    values = [r[key] for r in rows if r[key] is not None]
    return float(np.mean(values)) if values else None


def summarize(
    rows, levels, exchange, *, cost_per_request=False
):
    summaries = {}
    for level in levels:
        group = [r for r in rows if r["level"] == level]
        ok = [r for r in group if r["error"] is None]
        times = [r["latency_s"] for r in ok]
        dollars = (
            sum(
                r["input_tokens"] * INPUT_RATE
                + r["output_tokens"] * OUTPUT_RATE
                for r in ok
            )
            / 1_000_000
        )
        cost_denominator = (
            len(group) if cost_per_request else len(ok)
        )
        usd = dollars * 1000 / cost_denominator if ok else None
        summary = dict(
            requests=len(group),
            responses=len(ok),
            correct=sum(r["correct"] for r in ok),
            median_s=float(np.median(times)) if ok else None,
            p90_s=float(np.percentile(times, 90))
            if ok
            else None,
            max_s=max(times, default=None),
            mean_input=mean(ok, "input_tokens"),
            mean_output=mean(ok, "output_tokens"),
            mean_thinking=mean(ok, "thinking_tokens"),
            thinking_reported=sum(
                r["thinking_tokens"] is not None for r in ok
            ),
            max_tokens_stops=sum(
                r["stop_reason"] == "max_tokens" for r in ok
            ),
            observed_usd=dollars,
            usd_per_1000=usd,
            krw_per_1000=usd * exchange if ok else None,
        )
        summary["accuracy"] = (
            summary["correct"] / len(group) if group else None
        )
        summary["wrong_counts"] = {
            kind: sum(r.get("wrong_type") == kind for r in ok)
            for kind in WRONG_TYPES
        }
        summary["by_kind"] = {}
        for kind in KINDS:
            subset = [r for r in group if r["kind"] == kind]
            summary["by_kind"][kind] = dict(
                requests=len(subset),
                correct=sum(r["correct"] for r in subset),
                mean_output=mean(
                    [r for r in subset if not r["error"]],
                    "output_tokens",
                ),
            )
        summaries[level] = summary
    baseline = summaries.get("low", {})
    for s in summaries.values():
        s["relative_to_low"] = {
            key: s[key] / baseline[key]
            if baseline.get(key) and s[key] is not None
            else None
            for key in (
                "accuracy",
                "median_s",
                "p90_s",
                "max_s",
                "mean_input",
                "mean_output",
                "mean_thinking",
                "usd_per_1000",
            )
        }
    return summaries


summaries = summarize(rows, levels, KRW_PER_USD)
excluded_tasks = {
    r["task_id"] for r in rows
    if r["stop_reason"] == "max_tokens"
}
filtered_rows = [
    r for r in rows if r["task_id"] not in excluded_tasks
]
filtered = dict(
    excluded_task_ids=sorted(excluded_tasks),
    by_level=summarize(
        filtered_rows, levels, KRW_PER_USD,
        cost_per_request=True,
    ),
)
meta = dict(
    started_utc=started,
    sdk_version=anthropic.__version__,
    model=MODEL,
    dry_run=False,
    levels=levels,
    reps=reps,
    max_workers=WORKERS,
    max_tokens=MAX_TOKENS,
    tasks=len(tasks),
    krw_per_usd=KRW_PER_USD,
    input_usd_per_mtok=INPUT_RATE,
    output_usd_per_mtok=OUTPUT_RATE,
)
write_json(
    RESULTS / "results.json",
    dict(
        run=meta,
        summary=summaries,
        summary_without_max_tokens_tasks=filtered,
        calls=rows,
    ),
)

## 정답률·시간·토큰·요금 표

low 대비 표는 각 지표를 low 단계 값으로 나눈 배수입니다. 분모가 0이거나 측정값이 없으면 대시로 표시합니다. 이어서 단계별 오답 유형과 `max_tokens` 종료 과제를 제외한 집계를 표시합니다.

In [ ]:
def width(text):
    return sum(
        2 if unicodedata.east_asian_width(c) in "WF" else 1
        for c in text
    )


def print_detail(message):
    # 제어 문자를 공백으로 바꾸고 64칸에서 줄을 나눈다.
    line = ""
    for c in message:
        c = c if c.isprintable() else " "
        if width(line) + width(c) > 64:
            print(line)
            line = ""
        line += c
    print(line)


def table(headers, rows):
    cells = [headers, *rows]
    widths = [
        max(width(str(v)) for v in column)
        for column in zip(*cells)
    ]
    for row in cells:
        print_detail(
            "  ".join(
                str(cell) + " " * (w - width(str(cell)))
                for cell, w in zip(row, widths)
            ).rstrip()
        )


def fmt(value, places=1):
    return "—" if value is None else f"{value:.{places}f}"


def print_summary(summaries):
    print("\n정답 수 / 전체 요청")
    rows = []
    for level, s in summaries.items():
        counts = [(s["correct"], s["requests"])]
        counts.extend(
            (v["correct"], v["requests"])
            for v in s["by_kind"].values()
        )
        rows.append(
            [
                level,
                *(f"{n}/{d}" for n, d in counts),
                fmt(s["accuracy"] * 100),
            ]
        )
    table(["단계", "전체", "분류", "계산", "코딩", "%"], rows)

    print(f"\n응답 시간(초) · 동시 호출 최대 {WORKERS}개")
    table(
        ["단계", "중앙값", "p90", "최대", "토큰한도 종료"],
        [
            [
                level,
                fmt(s["median_s"], 2),
                fmt(s["p90_s"], 2),
                fmt(s["max_s"], 2),
                s["max_tokens_stops"],
            ]
            for level, s in summaries.items()
        ],
    )
    print("\n응답당 평균 토큰 · 출력에 추론 포함")
    table(
        ["단계", "입력", "출력", "추론", "추론 보고/응답"],
        [
            [
                level,
                fmt(s["mean_input"]),
                fmt(s["mean_output"]),
                fmt(s["mean_thinking"]),
                f"{s['thinking_reported']}/{s['responses']}",
            ]
            for level, s in summaries.items()
        ],
    )
    print("\n유형별 평균 출력 토큰")
    table(
        ["단계", "분류", "계산", "코딩"],
        [
            [
                level,
                *(
                    fmt(s["by_kind"][k]["mean_output"])
                    for k in KINDS
                ),
            ]
            for level, s in summaries.items()
        ],
    )
    print("\n응답 1,000건 환산 요금")
    table(
        ["단계", "달러", "원", "low 대비(배)"],
        [
            [
                level,
                fmt(s["usd_per_1000"], 4),
                fmt(s["krw_per_1000"], 2),
                fmt(s["relative_to_low"]["usd_per_1000"], 2),
            ]
            for level, s in summaries.items()
        ],
    )
    print("\nlow 대비 배수")
    table(
        [
            "단계",
            "정답률",
            "중앙초",
            "p90초",
            "최대초",
            "입력",
            "출력",
            "추론",
        ],
        [
            [
                level,
                *(
                    fmt(s["relative_to_low"][k], 2)
                    for k in (
                        "accuracy",
                        "median_s",
                        "p90_s",
                        "max_s",
                        "mean_input",
                        "mean_output",
                        "mean_thinking",
                    )
                ),
            ]
            for level, s in summaries.items()
        ],
    )


def print_diagnostics(summaries, filtered):
    print("\n단계별 오답 유형")
    table(
        ["단계", *WRONG_TYPES.values()],
        [
            [
                level,
                *(
                    s["wrong_counts"][kind]
                    for kind in WRONG_TYPES
                ),
            ]
            for level, s in summaries.items()
        ],
    )

    print("\nmax_tokens로 끝난 응답이 있는 과제를 뺀 집계")
    print_detail(
        "제외 과제 ID  "
        + (", ".join(filtered["excluded_task_ids"]) or "없음")
    )
    print("요금은 요청 1,000건 기준 · 배수는 low 요금 대비")
    table(
        [
            "단계", "정답/요청", "평균 출력", "평균 추론",
            "달러", "원", "배수",
        ],
        [
            [
                level,
                f'{s["correct"]}/{s["requests"]}',
                fmt(s["mean_output"]),
                fmt(s["mean_thinking"]),
                fmt(s["usd_per_1000"], 4),
                fmt(s["krw_per_1000"], 2),
                fmt(s["relative_to_low"]["usd_per_1000"], 2),
            ]
            for level, s in filtered["by_level"].items()
        ],
    )


print_summary(summaries)
print_diagnostics(summaries, filtered)

## 그래프 글꼴

한국어 그래프에는 Google Fonts의 NanumGothic TTF를 사용합니다(SIL Open Font License). 처음 실행할 때 `cache/fonts/`에 내려받아 Matplotlib에 등록합니다. [글꼴과 라이선스](https://github.com/google/fonts/tree/main/ofl/nanumgothic)

In [ ]:
def chart_font():
    path = CACHE / "fonts" / "NanumGothic-Regular.ttf"
    try:
        if not path.is_file():
            response = requests.get(FONT_URL, timeout=25)
            response.raise_for_status()
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(response.content)
        font_manager.fontManager.addfont(str(path))
        font = font_manager.FontProperties(fname=str(path))
        return font.get_name()
    except (OSError, RuntimeError, requests.RequestException):
        print("경고: 나눔고딕 대신 설치된 글꼴을 사용합니다.")
        names = font_manager.fontManager.get_font_names()
        for name in ("AppleGothic", "Malgun Gothic",
                     "Noto Sans CJK KR"):
            if name in names:
                return name
        print("경고: 기본 글꼴에서 한글이 빠질 수 있습니다.")
        return "DejaVu Sans"


langs = ("ko", "en") if CHART_LANG == "both" else (CHART_LANG,)
font = chart_font() if "ko" in langs else None

## effort 단계별 그래프

출력 토큰 구성, 요금과 정답 수의 관계, 응답 시간 분포를 그립니다. 그림은 선택한 언어별로 화면에 표시하고 `results/`에 PNG로 저장합니다.

In [ ]:
def plot_results(rows, summaries, language):
    levels = list(summaries)
    positions = np.arange(len(levels))
    colors = ("#8CB7B0", "#365F73", "#A6ADB2")
    ko = language == "ko"
    suffix = "_" + language

    def panel(xlabel, ylabel):
        fig, ax = plt.subplots(
            figsize=(6.5, 4), layout="constrained"
        )
        ax.set(xlabel=xlabel, ylabel=ylabel)
        ax.set_axisbelow(True)
        ax.grid(axis="y", alpha=0.20)
        ax.spines[["top", "right"]].set_visible(False)
        return fig, ax

    def save(fig, name):
        fig.savefig(
            RESULTS / f"{name}{suffix}.png",
            dpi=150,
            metadata={"Description": MODEL},
        )
        plt.show()
        plt.close(fig)

    effort = "추론 노력 단계" if ko else "Effort"
    with plt.rc_context(
        {
            "font.family": font if ko else "DejaVu Sans",
            "axes.unicode_minus": False,
        }
    ):
        fig, ax = panel(
            effort,
            "평균 출력 토큰" if ko else "Mean output tokens",
        )
        # 추론 토큰이 누락된 단계는 총 출력만 표시한다.
        for i, s in enumerate(summaries.values()):
            total = s["mean_output"]
            if total is None:
                continue
            if s["thinking_reported"] == s["responses"]:
                base = total - s["mean_thinking"]
                ax.bar(
                    i,
                    base,
                    color=colors[0],
                    label="추론 외" if ko else "Other output",
                )
                ax.bar(
                    i,
                    s["mean_thinking"],
                    bottom=base,
                    color=colors[1],
                    label="추론" if ko else "Thinking",
                )
            else:
                ax.bar(
                    i,
                    total,
                    color=colors[2],
                    label="총 출력" if ko else "Total output",
                )
        handles, labels = ax.get_legend_handles_labels()
        series = dict(zip(labels, handles))
        if len(series) > 1:
            ax.legend(
                series.values(), series.keys(), frameon=False
            )
        ax.set_xticks(positions, levels)
        save(fig, "fig1_output_tokens_by_effort")

        n = summaries[levels[0]]["requests"]
        fig, ax = panel(
            "요청 1,000건당 요금 (원)"
            if ko
            else "Cost per 1,000 requests (KRW)",
            f"정답 수 (/{n})"
            if ko
            else f"Correct answers (/{n})",
        )
        for level, s in summaries.items():
            if s["krw_per_1000"] is None:
                continue
            x, y = s["krw_per_1000"], s["correct"]
            ax.scatter(x, y, color=colors[1])
            below = level == "medium"
            ax.annotate(
                level,
                (x, y),
                xytext=(0, -7 if below else 7),
                textcoords="offset points",
                ha="center",
                va="top" if below else "baseline",
                fontsize=9,
            )
        ax.margins(x=0.12)
        ax.set_ylim(-0.04 * n, n * 1.13)
        ticks = MaxNLocator(integer=True, nbins=6).tick_values(
            0, n
        )
        ax.set_yticks(
            sorted({0, n, *(v for v in ticks if 0 < v < n)})
        )
        save(fig, "fig2_cost_vs_accuracy")

        fig, ax = panel(
            effort,
            "응답 시간 (초, 로그 눈금)"
            if ko
            else "Latency (s, log scale)",
        )
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(
            LogLocator(base=10, subs=(1, 2, 3, 5))
        )
        ax.yaxis.set_major_formatter(
            lambda value, _: f"{value:g}"
        )
        ax.yaxis.set_minor_formatter(NullFormatter())
        plotted = False
        for i, level in enumerate(levels):
            times = [
                r["latency_s"]
                for r in rows
                if r["level"] == level and not r["error"]
            ]
            times = [t for t in times if t > 0]
            if times:
                ax.boxplot(
                    times,
                    positions=[i],
                    widths=0.5,
                    patch_artist=True,
                    boxprops={"facecolor": colors[0]},
                    medianprops={"color": colors[1]},
                )
                plotted = True
        if not plotted:
            ax.set_ylim(0.1, 10)
        ax.set_xticks(positions, levels)
        ax.set_xlim(-0.6, len(levels) - 0.4)
        save(fig, "fig3_latency_by_effort")


for lang in langs:
    plot_results(rows, summaries, lang)

## 오답과 실패 호출

오답은 과제 ID·단계·반복·오답 유형·응답 마지막 줄을 표시합니다. 호출 실패에는 같은 식별 정보와 오류 종류를 표시합니다.

In [ ]:
wrong = [r for r in rows if not r["error"] and not r["correct"]]
if wrong:
    print(f"\n오답 {len(wrong)}건 (ID 단계 반복 유형 마지막 줄)")
    for r in wrong:
        label = WRONG_TYPES[r["wrong_type"]]
        if "format_value_correct" in r:
            label += (
                " (값 일치)" if r["format_value_correct"]
                else " (값 불일치)"
            )
        elif (
            r["wrong_type"] == "no_answer"
            and r["stop_reason"] == "max_tokens"
        ):
            label += " (max_tokens)"
        last_line = r["last_line"] or "—"
        print_detail(
            f"{r['task_id']} {r['level']} {r['rep']} "
            f"{label} {last_line}"
        )
failed = [r for r in rows if r["error"]]
print(f"\n실패 호출 {len(failed)}건")
for r in failed:
    print_detail(
        f"{r['task_id']} {r['level']} {r['rep']} {r['error']}"
    )